# Severstal Steel Defect Detection (Zero-Shot IVFPQ Pipeline)

This notebook demonstrates the complete industrial anomaly detection and multi-class segmentation pipeline:
1. **Frozen DINOv2 (ViT-B/14)** patch feature extraction ($16 \times 16 = 256$ spatial tokens per image).
2. **Quantized FAISS IndexIVFPQ** memory bank compression (192x compression ratio).
3. **Zero-Shot $k$-NN Majority Voting** for defect classification and segmentation.
4. **High-Resolution Industrial Visualization** color-coded by defect classes (1 to 4).

In [ ]:
import os
import sys
# Ensure src is discoverable from notebooks directory
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

from src.rle_utils import rle_to_mask, mask_to_rle
from src.dataset import SeverstalDataset
from src.evaluate import DefectInspector, CLASS_COLORS, CLASS_NAMES

## 1. Dataset Loading and Ground Truth Inspection

In [ ]:
dataset = SeverstalDataset(
    img_dir='../data/severstal/train_images',
    csv_path='../data/severstal/train.csv'
)
print(f"Total annotated images in training set: {len(dataset)}")

# Find a defective image for inspection
defect_idx = next(i for i, img_id in enumerate(dataset.image_ids) if dataset.image_to_rle.get(img_id, {}))
sample = dataset[defect_idx]
print(f"Inspecting Image ID: {sample['image_id']}")
print(f"Image Tensor Shape: {sample['image'].shape}")
print(f"16x16 Patch Mask Shape: {sample['mask_16x16'].shape}")
print(f"Full Resolution Mask Shape: {sample['full_mask'].shape}")

## 2. Zero-Shot IVFPQ Defect Inspector Inference

In [ ]:
inspector = DefectInspector(
    index_path='../data/severstal_ivfpq.index',
    labels_path='../data/severstal_labels.npy',
    k=5
)

img_path = os.path.join('../data/severstal/train_images', sample['image_id'])
gt_mask = sample['full_mask'].numpy()

results = inspector.evaluate_image(
    image_path_or_tensor=img_path,
    ground_truth_mask=gt_mask,
    output_path='../results/demo_notebook_inspection.png',
    image_id=sample['image_id']
)

print("Inspection Metrics:", results["metrics"])

## 3. Visualizing Segmentation Results

In [ ]:
result_img = Image.open('../results/demo_notebook_inspection.png')
plt.figure(figsize=(16, 10), dpi=150)
plt.imshow(result_img)
plt.axis('off')
plt.title('End-to-End Defect Segmentation Results', fontsize=14, fontweight='bold')
plt.show()